### Fase 1: Transformando Canciones en "Canastas"
El algoritmo Apriori no lee DataFrames normales; necesita una lista de transacciones. 
En nuestro caso, cada **Canción** es una transacción, y los **Artistas** que participan en ella son los artículos de la canasta.

In [3]:
# Instalar si es necesario: !pip install mlxtend
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# 1. INGESTA DE DATOS
url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/Most%20Streamed%20Spotify%20Songs%202024.csv"
df_spotify = pd.read_csv(url, encoding='latin-1')

# 2. LIMPIEZA: Eliminar nulos de la columna de artistas
df_spotify = df_spotify.dropna(subset=['Artist'])

# 3. TRANSFORMACIÓN: Separar los artistas que colaboran en cada canción
# Asumimos que las colaboraciones están separadas por coma (ej. "Bad Bunny, Feid")
# Convertimos esa columna en una verdadera lista de Python
lista_colaboraciones = df_spotify['Artist'].apply(lambda x: [artista.strip() for artista in x.split(',')]).tolist()

# Muestra rápida de cómo se ven nuestras "canastas"
print("Muestra de las primeras 5 colaboraciones:")
for i in range(5):
    print(f"Canción {i+1}: {lista_colaboraciones[i]}")


Muestra de las primeras 5 colaboraciones:
Canción 1: ['Tommy Richman']
Canción 2: ['Kendrick Lamar']
Canción 3: ['Artemas']
Canción 4: ['Miley Cyrus']
Canción 5: ['Eminem']


In [6]:

# Utiliza TransactionEncoder para convertir 'lista_colaboraciones' en una matriz booleana (One-Hot)
# Guárdala en un DataFrame llamado 'df_transacciones'

te = TransactionEncoder()
matriz_bool = te.fit_transform(lista_colaboraciones)
df_transacciones = pd.DataFrame(matriz_bool, columns=te.columns_)

print("<--- MATRIZ TRANSACCIONAL --->")
display(df_transacciones)

<--- MATRIZ TRANSACCIONAL --->


,"""XY""",$OHO BANI,$uicideboy$,&ME,(G)I-DLE,*NSYNC,.diedlonely,10CM,13 Organisï¿,1da Banton,...,ýýýýýýýýýý,ýýýýýýýýýý ýýýýýýýý,ýýýýýýýýýýýý,ýýýýýýýýýýýý ýýýýýýýýýýýýýý,ýýýýýýýýýýýý ýýýýýýýýýýýýýýýýýýýýýýýýýýý,ýýýýýýýýýýýýýýý ýýýýýýýýýýýýýýý,ýýýýýýýýýýýýýýý(ýýýýýýýýýýýýýýý),ýýýýýýýýýýýýýýýýýý,ýýýýýýýýýýýýýýýýýýýýý ýýýýýýýýýýýýýýýýýýýýý,ýýýýýýýýýýýýýýýýýýýýýýýý
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4590,False,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4591,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4592,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4593,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


### Fase 2: Aplicando Apriori
Ahora que tenemos la matriz con ceros y unos, extraigamos los conjuntos de artistas que colaboran frecuentemente.
Debido a que hay miles de artistas solistas, las colaboraciones son estadísticamente raras. Usaremos un `min_support` muy bajo para capturar las colaboraciones.

In [11]:
# --- RETO PARA EL ALUMNO AQUÍ ---

# 1. Encuentra los Itemsets Frecuentes usando un soporte mínimo del 0.005 (0.5% de las canciones)
itemsets_frecuentes = apriori(df_transacciones, min_support=0.0005, use_colnames=True)

# 2. Genera las Reglas de Asociación filtrando por la métrica 'lift' con un mínimo de 1.2
reglas_musicales = association_rules(itemsets_frecuentes, metric="lift", min_threshold=1.2)

# 3. Muestra las reglas resultantes ordenadas por 'confidence' de mayor a menor
display(reglas_musicales.sort_values(by="confidence", ascending=False))

print("Itemsets por tamaño:", itemsets_frecuentes['itemsets'].apply(len).value_counts().to_dict())
print("Canciones con 2+ artistas:", sum(len(set(c)) >= 2 for c in lista_colaboraciones))
print("Umbral en canciones:", 0.005 * len(df_transacciones))

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(The Creator),(Tyler),0.001523,0.001523,0.001523,1.0,656.428571,1.0,0.001521,inf,1.0,1.0,1.0,1.0
1,(Tyler),(The Creator),0.001523,0.001523,0.001523,1.0,656.428571,1.0,0.001521,inf,1.0,1.0,1.0,1.0


Itemsets por tamaño: {1: 381, 2: 1}
Canciones con 2+ artistas: 7
Umbral en canciones: 22.975


### 🧠 Fase 3: Interpretación de Negocio (El Entregable)

Como Científico de Datos, no basta con entregar una tabla. Debes explicarle los hallazgos al Director Creativo. Revisa tu DataFrame `reglas_musicales` y responde las siguientes 4 preguntas:

**1. Análisis de Soporte (El Volumen):**
Menciona a los dos artistas que conforman la regla con el mayor **Soporte**. ¿Qué significa este número en términos simples para el negocio?

> **Tu Respuesta:** No hay ningun artista que conforme la regla, aunque supuestamente existen 7 canciones con más de 2 artistas. Disminuyendo el soporte mínimo a 0.0005 se nos devuelve Tyler y The Creator. El problema es que no son dos artistas, el nombre del artista es "Tyler, The Creator". Así que no estoy seguro de como responder a la pregunta. 

**2. Análisis de Confianza (La Dirección):**
Busca una regla que tenga una **Confianza** mayor al 0.5 (50%). Traduce esta regla matemática a una frase que el director pueda entender (Ej. "Si contratamos al artista A, existe un X% de probabilidad de que el artista B también deba estar en el track para que sea el éxito que registramos").

> **Tu Respuesta:** Con los resultados del dataset, no se encontró a ninguna regla, por lo que no hay una colaboración entre artistas que pueda tener el éxito que registramos. 

**3. Análisis de Lift (El Impulso):**
Busca la regla con el **Lift** más alto de toda tu tabla. 
¿Por qué esta métrica es más confiable que el Soporte para decidir si estos dos artistas *realmente* tienen química musical o si solo fue una coincidencia estadística?

> **Tu Respuesta:** Sin reglas no puedo comparar el lift. Pero se podría confiar más en el lift que en el soporte porque mide cuanto más seguido aparecen dos items juntos de lo que ocurriría si fueran independientes. Por otro lado, el soporte solo cuenta que tanto aparecen juntos y el resultado. 

**4. Toma de Decisión:**
Basado puramente en las reglas matemáticas que descubriste, si la disquera solo tiene presupuesto para juntar a dos artistas en el estudio la próxima semana, ¿qué pareja elegirías y por qué? (Justifica usando las 3 métricas de oro).

> **Tu Respuesta:** No puedo recomendar ninguna pareja porque nignuna cumple los umbrales. El dataset tiene muchas más canciones de un solo artista, por lo que no se tiene un buen historial de colaboraciones para respaldar una elección.